In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
df_X3 = pd.read_csv("homework_3.1.csv")

In [5]:
# Create the regression variables
df_X3['D'] = (df_X3['time'] >= 50).astype(int)
df_X3['time_centered'] = df_X3['time'] - 50
df_X3['D_time_centered'] = df_X3['D'] * df_X3['time_centered']

# Run regressions for each value column
for val in ['value1', 'value2', 'value3']:
    print(f"\n=================== Analysis for {val} ===================")
    
    # Model 1: Discontinuity in value only
    X1 = sm.add_constant(df_X3[['time_centered', 'D']])
    model1 = sm.OLS(df_X3[val], X1).fit()
    print("--- Model 1: Value Discontinuity Only ---")
    print(model1.summary().tables[1])
    
    # Model 2: Value and Slope Discontinuity
    X2 = sm.add_constant(df_X3[['time_centered', 'D', 'D_time_centered']])
    model2 = sm.OLS(df_X3[val], X2).fit()
    print("--- Model 2: Value and Slope Discontinuity ---")
    print(model2.summary().tables[1])


=================== Analysis for value1 ===================
--- Model 1: Value Discontinuity Only ---
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.7488      0.277      6.310      0.000       1.199       2.299
time_centered     0.0439      0.008      5.168      0.000       0.027       0.061
D                 0.8508      0.490      1.737      0.086      -0.121       1.823
--- Model 2: Value and Slope Discontinuity ---
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4059      0.274      1.479      0.142      -0.139       0.951
time_centered      -0.0088      0.009     -0.940      0.349      -0.027       0.010
D                   0.9035      0.382      2.362      0.020       0.144       1.663
D_time_centered     

In [6]:
# Create the features for Interrupted Time Series Analysis
df_X3['D'] = (df_X3['time'] >= 50).astype(int)
df_X3['time_centered'] = df_X3['time'] - 50
df_X3['D_time_centered'] = df_X3['D'] * df_X3['time_centered']

# Add constant for the regression intercept
X = sm.add_constant(df_X3[['time_centered', 'D', 'D_time_centered']])

# Run the regression for each dataset and print the derivative discontinuity metric
for val in ['value1', 'value2', 'value3']:
    model = sm.OLS(df_X3[val], X).fit()
    coef = model.params['D_time_centered']
    t_stat = model.tvalues['D_time_centered']
    p_val = model.pvalues['D_time_centered']
    
    print(f"{val}: Slope Change (Beta_3) = {coef:.4f}, t-statistic = {t_stat:.3f}, p-value = {p_val:.5f}")

value1: Slope Change (Beta_3) = 0.1053, t-statistic = 7.951, p-value = 0.00000
value2: Slope Change (Beta_3) = 0.0369, t-statistic = 2.567, p-value = 0.01181
value3: Slope Change (Beta_3) = 0.0507, t-statistic = 3.857, p-value = 0.00021


In [8]:
# Load datasets
df_a = pd.read_csv('homework_3.2.a.csv')
df_b = pd.read_csv('homework_3.2.b.csv')

# Estimate DiD model for Dataset A (homework_3.2.a.csv)
model_a = smf.ols('outcome1 ~ group1 + time1 + group1:time1', data=df_a).fit()
print("=== Dataset A (homework_3.2.a.csv) ===")
print(model_a.summary().tables[1])

# Estimate DiD model for Dataset B (homework_3.2.b.csv)
model_b = smf.ols('outcome2 ~ group2 + time2 + group2:time2', data=df_b).fit()
print("\n=== Dataset B (homework_3.2.b.csv) ===")
print(model_b.summary().tables[1])

=== Dataset A (homework_3.2.a.csv) ===
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -0.0258      0.031     -0.829      0.408      -0.087       0.035
group1           1.9863      0.044     44.928      0.000       1.900       2.073
time1            1.4272      0.044     32.315      0.000       1.341       1.514
group1:time1     0.6858      0.063     10.970      0.000       0.563       0.809

=== Dataset B (homework_3.2.b.csv) ===
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.1021      0.073      1.392      0.164      -0.042       0.246
group2           1.8477      0.104     17.770      0.000       1.644       2.052
time2            1.2655      0.104     12.183      0.000       1.062       1.469
group2:time2     1.3499      0

In [9]:
# Fit Differences-in-Differences models
model_a = smf.ols('outcome1 ~ group1 + time1 + group1:time1', data=df_a).fit()
model_b = smf.ols('outcome2 ~ group2 + time2 + group2:time2', data=df_b).fit()

# Extract statistics for the treatment effect (interaction terms)
print("=== Dataset A (homework_3.2.a.csv) ===")
print(f"Treatment Effect Coef : {model_a.params['group1:time1']:.4f}")
print(f"Standard Error        : {model_a.bse['group1:time1']:.4f}")
print(f"t-statistic           : {model_a.tvalues['group1:time1']:.3f}")
print(f"p-value               : {model_a.pvalues['group1:time1']:.5e}")

print("\n=== Dataset B (homework_3.2.b.csv) ===")
print(f"Treatment Effect Coef : {model_b.params['group2:time2']:.4f}")
print(f"Standard Error        : {model_b.bse['group2:time2']:.4f}")
print(f"t-statistic           : {model_b.tvalues['group2:time2']:.3f}")
print(f"p-value               : {model_b.pvalues['group2:time2']:.5e}")

=== Dataset A (homework_3.2.a.csv) ===
Treatment Effect Coef : 0.6858
Standard Error        : 0.0625
t-statistic           : 10.970
p-value               : 1.64005e-26

=== Dataset B (homework_3.2.b.csv) ===
Treatment Effect Coef : 1.3499
Standard Error        : 0.1470
t-statistic           : 9.180
p-value               : 2.43244e-19


In [10]:
# Fit the Differences-in-Differences regression model
model_b = smf.ols('outcome2 ~ group2 + time2 + group2:time2', data=df_b).fit()

# Print the treatment effect coefficient (interaction term)
treatment_effect = model_b.params['group2:time2']
print(f"Treatment Effect for Group 2: {treatment_effect}")

Treatment Effect for Group 2: 1.349858924679673
